<a href="https://colab.research.google.com/github/shaify-cloud/ai-mentor-portfolio/blob/main/Day8_Memory_FineTune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q langchain-community langchain-google-vertexai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 31.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.7/118.7 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 354.8/354.8 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 34.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.3/66.3 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.7/44.7 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 375.2/375.2 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.

In [ ]:
import json, pathlib
from datetime import datetime

MEM = pathlib.Path('memory.json')

def load_memory(student_id: str) -> list:
    """Load conversation history for a student, capped at last 20 turns."""
    if not MEM.exists():
        return []
    data = json.loads(MEM.read_text())
    return data.get(student_id, [])

def save_message(student_id: str, role: str, content: str):
    """Append a message; cap student's history at last 20 turns."""
    data = json.loads(MEM.read_text()) if MEM.exists() else {}
    data.setdefault(student_id, []).append({
        'role': role,
        'content': content,
        'ts': datetime.now().isoformat(),
    })
    data[student_id] = data[student_id][-20:]   # cap at last 20
    MEM.write_text(json.dumps(data, indent=2))

# Test
save_message('S001', 'user', 'What does TCS Digital want?')
save_message('S001', 'assistant', 'Java + DSA + CGPA 7+')
save_message('S001', 'user', 'And what about Cognizant?')
save_message('S001', 'assistant', 'Java + Python + DSA + CGPA 6.5+')

print('Memory for S001:')
for msg in load_memory('S001'):
    print(f'  [{msg["role"]}] {msg["content"]}')

# Test cap at 20
for i in range(25):
    save_message('S002', 'user', f'message {i}')
print(f'\nS002 has {len(load_memory("S002"))} messages (should be 20)')

Memory for S001:
  [user] What does TCS Digital want?
  [assistant] Java + DSA + CGPA 7+
  [user] And what about Cognizant?
  [assistant] Java + Python + DSA + CGPA 6.5+

S002 has 20 messages (should be 20)


In [ ]:
def rag_with_memory(student_id: str, question: str) -> str:
    # Retrieve recent conversation context
    history = load_memory(student_id)
    history_text = '\n'.join(f'{m["role"]}: {m["content"]}' for m in history[-6:])  # last 6 turns

    # Augment the question with history
    augmented = f'Conversation so far:\n{history_text}\n\nNew question: {question}'

    result = qa.invoke({'query': augmented})
    answer = result['result']

    # Save to memory
    save_message(student_id, 'user', question)
    save_message(student_id, 'assistant', answer)

    return answer

# Test
print(rag_with_memory('S001', 'And what is the package range?'))

NameError: name 'qa' is not defined